# Virtual Mouse / Air Canvas - Phase 1

## Environment setup & verification

Phase 1 has one job: get the libraries installed and **prove they work**
before writing any real code. By the end of this notebook you will have:

- the four core libraries installed at known-good versions
- a confirmed working MediaPipe (the part that breaks most often)
- your screen size read correctly (needed later to move the cursor)

Run the cells **top to bottom**. Where a cell needs a one-time action
(install, restart) it says so clearly.

---
### The single most common setup failure
The newest MediaPipe (0.10.22 and later) **removed** the
`mediapipe.solutions.hands` API that this whole project uses. A plain
`pip install mediapipe` gives you the newest one, and later your import
dies with:

`module 'mediapipe' has no attribute 'solutions'`

We avoid this by pinning `mediapipe==0.10.21` below. The matching NumPy and
OpenCV versions are pinned with it so all three agree.

## Step 0 - Check your Python version

MediaPipe needs Python **3.9 - 3.12**. If this prints 3.13 or higher,
MediaPipe will not install - make a new environment with Python 3.12.

In [1]:
import sys
print("Python:", sys.version.split()[0])
major, minor = sys.version_info[:2]
ok = (major == 3 and 9 <= minor <= 12)
print("Version OK for MediaPipe:" , ok)
if not ok:
    print("  -> Use Python 3.9-3.12. Create a fresh venv with a supported Python.")

Python: 3.12.0
Version OK for MediaPipe: True


## Step 1 - Install the core libraries (run ONCE)

Uncomment the line and run this cell **one time**. After it finishes,
**restart the kernel** (Kernel -> Restart) so the fresh installs load,
then continue from Step 2.

What each library does:
- **opencv-python** - camera capture and image operations
- **mediapipe** - the 21-point hand detector (the heart of the project)
- **numpy** - fast math for coordinates and smoothing
- **pyautogui** - moves the real mouse cursor (used from Phase 3 on)

In [2]:
# !pip install opencv-python==4.9.0.80 mediapipe==0.10.21 numpy==1.26.4 pyautogui==0.9.54

## Step 2 - Import everything

If any import fails here, the install above did not complete or the kernel
was not restarted. Re-run Step 1, restart the kernel, and try again.

In [18]:
!pip uninstall mediapipe==0.10.35

^C


In [1]:
!pip uninstall mediapipe
!pip install mediapipe==0.10.21
!pip install pyautogui==0.9.54
!pip install numpy==1.26.4

^C


In [2]:
import cv2
import numpy as np
import mediapipe as mp
import pyautogui

print("All four libraries imported successfully.")

All four libraries imported successfully.


## Step 3 - Print versions

These should match the pinned versions from Step 1.

In [3]:
print("OpenCV    :", cv2.__version__)
print("NumPy     :", np.__version__)
print("MediaPipe :", mp.__version__)
print("PyAutoGUI :", pyautogui.__version__)

OpenCV    : 4.11.0
NumPy     : 1.26.4
MediaPipe : 0.10.21
PyAutoGUI : 0.9.54


## Step 4 - The critical MediaPipe check

This is the check that catches the "no attribute 'solutions'" failure.
If the assert passes, you have the right MediaPipe and the project will work.

In [4]:
assert hasattr(mp, "solutions"), "Wrong MediaPipe - need 0.10.21 (see Step 1)."
assert hasattr(mp.solutions, "hands"), "MediaPipe Hands missing - need 0.10.21."

# Actually build the Hands model to be 100% sure it initialises.
hands = mp.solutions.hands.Hands(max_num_hands=1)
print("MediaPipe Hands initialised successfully - the vision engine works.")
hands.close()

MediaPipe Hands initialised successfully - the vision engine works.


## Step 5 - Read your screen size

PyAutoGUI reports the resolution we will later map hand movement onto.
On your display this should print your monitor's pixel size.

In [5]:
w, h = pyautogui.size()
print(f"Screen size: {w} x {h} pixels")
print("This is the area the cursor will be mapped across in Phase 3.")

Screen size: 1920 x 1080 pixels
This is the area the cursor will be mapped across in Phase 3.


## Step 6 - Run a tiny MediaPipe pipeline (no camera needed)

This feeds one blank frame through the full detect path. We are not looking
for a hand - just confirming `process()` runs end to end without errors.
A clean run here means the whole pipeline is ready.

In [6]:
test_frame = np.zeros((480, 640, 3), dtype=np.uint8)        # blank BGR image
rgb = cv2.cvtColor(test_frame, cv2.COLOR_BGR2RGB)           # MediaPipe wants RGB

hands = mp.solutions.hands.Hands(max_num_hands=1)
results = hands.process(rgb)
hands.close()

found = results.multi_hand_landmarks is not None
print("Pipeline ran end to end - OK")
print("Hand detected in blank frame:", found, "(expected: False)")
print("\nPhase 1 complete. Environment is ready for Phase 2.")

Pipeline ran end to end - OK
Hand detected in blank frame: False (expected: False)

Phase 1 complete. Environment is ready for Phase 2.
